In [2]:
# Install dependencies
!pip install onnxruntime torchaudio transformers pandas pydub tqdm --quiet
!apt-get install -y ffmpeg > /dev/null 2>&1

import os
import torch
import torchaudio
import pandas as pd
from pydub import AudioSegment
from transformers import AutoModel, pipeline
from tqdm.notebook import tqdm

# ==========================
# Load ASR and sentiment models
# ==========================
print("Loading ASR model...")
asr_model = AutoModel.from_pretrained(
    "ai4bharat/indic-conformer-600m-multilingual",
    trust_remote_code=True
)

print("Loading sentiment analysis model...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# ==========================
# Audio processing function
# ==========================
def load_audio_any_format(file_path, target_sample_rate=16000):
    """Load any audio format and convert to mono + target_sample_rate"""
    try:
        wav, sr = torchaudio.load(file_path)
    except:
        audio = AudioSegment.from_file(file_path)
        audio = audio.set_channels(1).set_frame_rate(target_sample_rate)
        samples = torch.tensor(audio.get_array_of_samples()).float().unsqueeze(0) / (1 << 15)
        return samples, target_sample_rate
    # Convert stereo → mono
    wav = torch.mean(wav, dim=0, keepdim=True)
    # Resample if needed
    if sr != target_sample_rate:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sample_rate)
        wav = resampler(wav)
    return wav, target_sample_rate

# ==========================
# Set your audio folder here
# ==========================
folder_path = "/content/Recordings"
audio_files = [f for f in os.listdir(folder_path) if f.lower().endswith(
    (".wav", ".mp3", ".flac", ".ogg", ".m4a"))]

results = []
target_sample_rate = 16000

# ==========================
# Process files with progress bar
# ==========================
for file in tqdm(audio_files, desc="Processing Audio Files"):
    file_path = os.path.join(folder_path, file)
    try:
        wav, sr = load_audio_any_format(file_path, target_sample_rate)

        # ASR inference → returns string directly
        transcript = asr_model(wav, lang="hi")

        # Sentiment analysis
        sentiment = sentiment_pipeline(transcript)[0]
        sentiment_result = f"{sentiment['label']} ({sentiment['score']:.2f})"

        results.append({
            "filename": file,
            "transcript": transcript,
            "sentiment": sentiment_result
        })

    except Exception as e:
        print(f"⚠️ Skipping {file} due to error: {e}")

# ==========================
# Save results to CSV
# ==========================
df = pd.DataFrame(results)
output_csv = "transcriptions_with_sentiment.csv"
df.to_csv(output_csv, index=False)
print(f"✅ All done! Saved results to {output_csv}")


Loading ASR model...


Fetching 403 files:   0%|          | 0/403 [00:00<?, ?it/s]

onnx__MatMul_8700:   0%|          | 0.00/4.19M [00:00<?, ?B/s]

onnx__MatMul_8689:   0%|          | 0.00/4.19M [00:00<?, ?B/s]

onnx__MatMul_8723:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cpu


Processing Audio Files:   0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

✅ All done! Saved results to transcriptions_with_sentiment.csv
